# TabDPT Classifier Artifact Inference — DIMER tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/tutorials/tabdpt_classifier_artifact_inference_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Layer6%2FTabDPT-ffcc4d?style=flat)](https://huggingface.co/Layer6/TabDPT)
[![Upstream](https://img.shields.io/badge/Upstream-layer6ai--labs%2FTabDPT--inference-181717?style=flat&logo=github&logoColor=white)](https://github.com/layer6ai-labs/TabDPT-inference)
[![arXiv](https://img.shields.io/badge/arXiv-2608.01400-b31b1b.svg)](https://arxiv.org/abs/2608.01400)

**Profile:** `ARTIFACT-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification `1.0`  
**Repository code revision exercised:** `69c37977eb0606070569066f95f27a7f7f9f3a1d`

This notebook consumes an **externally supplied** `tabdpt-dimer-context-v3` artifact (`artifact.json` + `training_context.parquet`). It validates the artifact before model reconstruction, restores training-fitted preprocessing and in-context support state, then scores **genuinely new input** without refitting preprocessing.

`TabDPTClassificationPipeline.load_artifact()` reconstructs serving state. TabDPT is an in-context learner: the restored support table conditions inference, but the immutable TabDPT weights are not gradient-trained or fine-tuned here.

**By the end of this notebook you will be able to:**
- validate an externally supplied DIMER v3 artifact using the repository's strict artifact validator;
- inspect its format, base-model identity, immutable revision, class order, fitted schema, runtime controls, and support-context digest;
- reconstruct the serving object from the artifact contract;
- validate and score a new unlabelled CSV without refitting training-fitted preprocessing;
- export predictions and provenance as machine-readable files.

**This notebook does not create its own artifact.** Produce the artifact in a separate E2E/DIMER execution and supply it here. It does not demonstrate gradient fine-tuning, model selection, calibrated probabilities, or production fitness.

References: [README](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/README.md), [model card](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/MODEL_CARD.md), [dataset specification](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/TABULAR_CLASSIFICATION_DATASET_SPEC.md), and [DIMER contract](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/DIMER_CONTRACT.md).


## Prerequisites and trust boundary

- **Runtime:** fresh Google Colab or compatible Jupyter; Python 3.11–3.13.
- **Accelerator:** GPU recommended; CPU supported but slower. The demonstrated path forces `use_flash=False` for Tesla T4 portability.
- **External inputs required:** `artifact.json`, `training_context.parquet`, and one new unlabelled CSV.
- **Expected new-input schema:** exactly the fitted feature columns declared by the artifact, plus any configured drop columns if applicable; the training target must not be supplied for scoring.
- **Privacy:** uploaded files remain in the notebook runtime unless you explicitly move/export them. Do not upload restricted data to a hosted environment without authorization.
- **Archive policy:** this notebook accepts individual files only, not ZIP/TAR archives, so archive extraction requirements are not applicable.
- **Operational controls:** inference uses the artifact's recorded `n_ensembles`, `context_size`, `batch_size`, `temperature`, and `seed`. Context support may be subsampled by TabDPT according to those controls.

**Trust boundary:** manifest structure, file-size checks, and SHA-256 establish internal consistency, not sender authenticity. The v3 artifact is data-only JSON + Parquet; the base checkpoint is acquired separately from the repository-pinned immutable model revision and verified by SHA-256. Do not treat path safety or checksum agreement as proof that an artifact came from a trusted producer.

**Reproducibility boundary:** the artifact seed and inference seed control documented stochastic choices, including ensemble/context sampling. They do not guarantee bitwise-identical floating-point scores across different devices, CUDA/library builds, or kernels.


In [ ]:
import sys
if "torch" in sys.modules:
    raise RuntimeError(
        "Start from a fresh runtime: PyTorch is already imported. "
        "Install the pinned tutorial environment before importing core ML packages."
    )

REPO_REVISION = "69c37977eb0606070569066f95f27a7f7f9f3a1d"
REPO_DIR = "/content/tabdpt-classifier-pipeline"

!rm -rf "$REPO_DIR"
!git clone -q https://github.com/kurtvalcorza/tabdpt-classifier-pipeline.git "$REPO_DIR"
!git -C "$REPO_DIR" checkout -q "$REPO_REVISION"
!python -m pip install -q -r "$REPO_DIR/tutorials/requirements-colab.txt"
!python -m pip install -q --no-deps "$REPO_DIR"


## 1. Verify runtime and expected model identity

The next cell reports the effective runtime and the immutable TabDPT identity this repository accepts. Successful identity/checksum verification establishes the model bytes used; it is not evidence of model quality or deployment fitness.


In [ ]:
import importlib.metadata as mdlib
import platform
import sys
import torch

from tabdpt_classifier_pipeline import (
    TABDPT_HF_REPO,
    TABDPT_HF_REVISION,
    TABDPT_UPSTREAM_CODE_COMMIT,
    TABDPT_WEIGHT_FILENAME,
    TABDPT_WEIGHT_SHA256,
    TabDPTClassificationPipeline,
    resolve_tabdpt_weights,
    validate_dimer_artifact,
)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for package in ["tabdpt", "torch", "numpy", "pandas", "scikit-learn", "huggingface-hub", "pyarrow"]:
    print(f"{package}:", mdlib.version(package))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA runtime:", torch.version.cuda)
print("Tutorial settings: compile_model=False, use_flash=False")
print("Expected model:", TABDPT_HF_REPO)
print("Expected revision:", TABDPT_HF_REVISION)
print("Expected upstream code:", TABDPT_UPSTREAM_CODE_COMMIT)
print("Expected weight file:", TABDPT_WEIGHT_FILENAME)
print("Expected weight SHA-256:", TABDPT_WEIGHT_SHA256)

weights = resolve_tabdpt_weights()
print("Verified checkpoint path:", weights)


## 2. Supply and validate the external artifact

Upload exactly `artifact.json` and `training_context.parquet` from a **separate producing execution**. `validate_dimer_artifact()` runs before serving-state reconstruction and rejects the wrong format/task, model-provenance mismatch, malformed fitted preprocessing, inconsistent class/target/drop metadata, missing or invalid runtime controls, unsafe context paths, symlinks, oversized context files, declared-size mismatch, digest mismatch, and unexpected unlisted files.

Successful validation proves that the bundle is internally consistent with this repository's v3 contract. It does **not** authenticate the sender.


In [ ]:
from pathlib import Path
import shutil

ARTIFACT_DIR = Path("/content/external-tabdpt-artifact")
shutil.rmtree(ARTIFACT_DIR, ignore_errors=True)
ARTIFACT_DIR.mkdir(parents=True)

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError(
        f"Local Jupyter: place artifact.json and training_context.parquet in {ARTIFACT_DIR}, "
        "then replace only this acquisition cell with your local copy step."
    ) from exc

uploaded = files.upload()
expected = {"artifact.json", "training_context.parquet"}
received = {Path(name).name for name in uploaded}
if received != expected:
    raise ValueError(f"Upload exactly {sorted(expected)}; received {sorted(received)}")
for name, payload in uploaded.items():
    (ARTIFACT_DIR / Path(name).name).write_bytes(payload)

manifest_path = ARTIFACT_DIR / "artifact.json"
manifest, context_path = validate_dimer_artifact(manifest_path, strict_directory=True)

runtime_config = manifest["runtimeConfig"]
preprocessing = manifest["preprocessing"]
print("Artifact format:", manifest["format"])
print("Task:", manifest["taskType"])
print("Base model:", manifest["baseModel"]["repo"])
print("Base revision:", manifest["baseModel"]["revision"])
print("Weight SHA-256:", manifest["baseModel"]["sha256"])
print("Context file:", context_path.name)
print("Context size bytes:", context_path.stat().st_size)
print("Context SHA-256:", manifest["trainingContext"]["sha256"])
print("Classes:", manifest["classNames"])
print("Feature count:", len(preprocessing["encoder"]["featureColumns"]))
print("Runtime controls:", runtime_config)


## 3. Reconstruct the serving state

`load_artifact()` restores the fitted feature encoder and support context from the validated bundle. It does not refit preprocessing from the inference CSV. Internally, the upstream in-context estimator registers the restored support set; that operation is conditioning, not gradient training.

The immutable base checkpoint is verified again through repository code during reconstruction. Automatic substitution with another model identity is not part of this workflow.


In [ ]:
pipe = TabDPTClassificationPipeline.load_artifact(
    manifest_path,
    compile_model=False,
    use_flash=False,
    seed=runtime_config["seed"],
)
print("Restored target:", pipe.target_column)
print("Restored classes:", pipe.class_labels_)
print("Restored features:", pipe.feature_encoder.feature_columns)


## 4. Upload and validate genuinely new input

Upload one **unlabelled CSV** that was not used to produce the artifact. Duplicate headers are rejected before pandas can silently rename them. The notebook then verifies the target is absent and checks the effective feature schema against the fitted artifact state before model execution.

Unknown categorical values use the training-fitted unknown-category code rather than creating a new category map. The notebook reports them because schema-compatible category drift can still matter operationally.


In [ ]:
import csv
from collections import Counter
import pandas as pd

INPUT_DIR = Path("/content/tabdpt-inference-input")
shutil.rmtree(INPUT_DIR, ignore_errors=True)
INPUT_DIR.mkdir(parents=True)

uploaded_input = files.upload()
csv_names = [Path(name).name for name in uploaded_input if Path(name).suffix.lower() == ".csv"]
if len(uploaded_input) != 1 or len(csv_names) != 1:
    raise ValueError("Upload exactly one CSV containing new unlabelled records")
input_path = INPUT_DIR / csv_names[0]
input_path.write_bytes(next(iter(uploaded_input.values())))

with input_path.open("r", encoding="utf-8-sig", newline="") as handle:
    header = next(csv.reader(handle), None)
if not header:
    raise ValueError("Inference CSV is empty")
duplicates = sorted(name for name, count in Counter(header).items() if count > 1)
if duplicates:
    raise ValueError(f"Duplicate column names are not supported: {duplicates}")

new_data = pd.read_csv(input_path)
if pipe.target_column in new_data.columns:
    raise ValueError(f"Inference input must be unlabelled; remove target column {pipe.target_column!r}")

effective = new_data.drop(columns=pipe.drop_columns_, errors="ignore")
required = list(pipe.feature_encoder.feature_columns)
missing = [col for col in required if col not in effective.columns]
extra = [col for col in effective.columns if col not in required]
if missing or extra:
    raise ValueError(f"Feature schema mismatch; missing={missing}, extra={extra}")

for col, mapping in pipe.feature_encoder.category_maps.items():
    if col in effective.columns:
        observed = {str(v) for v in effective[col].dropna().tolist()}
        unseen = sorted(observed - set(mapping))
        if unseen:
            print(f"WARNING: {col!r} has {len(unseen)} unseen categorical value(s): {unseen[:10]}")

print("Validated new input shape:", new_data.shape)


## 5. Predict and export machine-readable results

The pipeline's default classification decision rule is `argmax` over the class-score columns returned by `predict_proba()`. Those outputs are model class scores normalized as probabilities by the upstream classifier, but this tutorial does **not** establish calibration; do not treat them as guaranteed confidence estimates.

Score columns preserve `pipe.class_labels_` ordering. `row_id` preserves the mapping from every output row to the uploaded input row.


In [ ]:
import hashlib
import json

inference_kwargs = {
    "n_ensembles": runtime_config["n_ensembles"],
    "context_size": runtime_config["context_size"],
    "batch_size": runtime_config["batch_size"],
    "temperature": runtime_config["temperature"],
    "seed": runtime_config["seed"],
}
print("Effective inference controls:", inference_kwargs)
print("Context note: context_size limits per-ensemble support context; TabDPT may seeded-subsample support rows when the saved context is larger.")

pred = pipe.predict(new_data, **inference_kwargs)
scores = pipe.predict_proba(new_data, **inference_kwargs)

result = pd.DataFrame({"row_id": new_data.index, "prediction": pred.to_numpy()})
for label in pipe.class_labels_:
    result[f"score_{label}"] = scores[label].to_numpy()

OUTPUT_DIR = Path("/content/tabdpt-artifact-inference-output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pred_path = OUTPUT_DIR / "predictions.csv"
prov_path = OUTPUT_DIR / "provenance.json"
result.to_csv(pred_path, index=False)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

provenance = {
    "notebookProfile": "ARTIFACT-INFERENCE",
    "notebookSpec": "1.0",
    "repositoryRevision": REPO_REVISION,
    "artifact": {
        "format": manifest["format"],
        "manifestSha256": sha256_file(manifest_path),
        "contextSha256": manifest["trainingContext"]["sha256"],
    },
    "model": manifest["baseModel"],
    "runtimeConfig": runtime_config,
    "runtime": {
        "python": sys.version.split()[0],
        "torch": mdlib.version("torch"),
        "tabdpt": mdlib.version("tabdpt"),
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "cuda": torch.version.cuda,
        "use_flash": False,
        "compile_model": False,
    },
    "input": {
        "filename": input_path.name,
        "sha256": sha256_file(input_path),
        "rows": len(new_data),
        "columns": list(new_data.columns),
    },
    "classOrder": list(pipe.class_labels_),
    "decisionRule": "argmax over class-score columns",
    "calibrationEstablished": False,
}
prov_path.write_text(json.dumps(provenance, indent=2) + "\n", encoding="utf-8")

print(result.head())
print("Predictions:", pred_path)
print("Provenance:", prov_path)


## Interpretation and troubleshooting

A successful run demonstrates that this exact notebook revision can validate a separately produced DIMER v3 TabDPT artifact, reconstruct its serving state from the declared support/preprocessing contract plus the pinned base model, validate genuinely new input, and produce machine-readable predictions with provenance.

It **does not prove** that the model is accurate, calibrated, fair, robust, secure for arbitrary domains, or production-ready. No evaluation metric is reported here because the required inference input is intentionally unlabelled. Evaluate task quality on appropriately labelled, leakage-safe data in the producing E2E workflow or an independent evaluation workflow.

Common failures are intentional safeguards: a provenance/digest/size error means the artifact must not be reconstructed; a schema mismatch means the new CSV does not match the fitted contract; a missing checkpoint/network error means the exact pinned model could not be acquired; memory pressure can be reduced only by changing serving controls through a newly produced artifact or an explicitly governed deployment configuration, not by silently changing the artifact's recorded semantics.

**Next steps:** compare performance on a domain-valid independent labelled test set, assess probability calibration if decisions depend on score magnitudes, and validate deployment-specific latency/memory on the actual serving hardware. Successful tutorial execution remains demonstration evidence, not production acceptance.
